In [14]:
import pandas as pd
import numpy as np
import folium
import os
from sklearn.cluster import DBSCAN
from IPython.display import display

In [15]:
df = pd.read_csv('food_delivery_orders.csv')
df

,Order_ID,Order_Date,Latitude,Longitude,Order_Value_INR
0,FD0405,2026-01-04,11.649742,78.165180,176
1,FD0419,2026-01-11,11.649736,78.166180,257
2,FD0433,2026-01-22,11.649439,78.165687,126
3,FD0151,2026-01-04,11.682398,78.118732,615
4,FD0071,2026-01-16,11.664635,78.145182,658
...,...,...,...,...,...
645,FD0084,2026-01-27,11.665054,78.146318,346
646,FD0490,2026-01-27,11.688853,78.160367,411
647,FD0278,2026-01-16,11.680930,78.119746,535
648,FD0483,2026-01-03,11.689971,78.158872,115


In [16]:
df.head()

,Order_ID,Order_Date,Latitude,Longitude,Order_Value_INR
0,FD0405,2026-01-04,11.649742,78.165180,176
1,FD0419,2026-01-11,11.649736,78.166180,257
2,FD0433,2026-01-22,11.649439,78.165687,126
3,FD0151,2026-01-04,11.682398,78.118732,615
4,FD0071,2026-01-16,11.664635,78.145182,658


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         650 non-null    object 
 1   Order_Date       650 non-null    object 
 2   Latitude         650 non-null    float64
 3   Longitude        650 non-null    float64
 4   Order_Value_INR  650 non-null    int64  
dtypes: float64(2), int64(1), object(2)
memory usage: 25.5+ KB


In [18]:
df.isnull().sum()

,0
Order_ID,0
Order_Date,0
Latitude,0
Longitude,0
Order_Value_INR,0


In [28]:
import folium
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from IPython.display import display
from branca.colormap import linear

# Ensure the Cluster column exists on the locations dataframe
if 'Cluster' not in locations.columns:
    coords = np.radians(locations[['Latitude', 'Longitude']].to_numpy())
    earth_radius_km = 6371.0088
    radius_km = 0.35
    model = DBSCAN(eps=radius_km / earth_radius_km, min_samples=12,
                   algorithm='ball_tree', metric='haversine')
    locations['Cluster'] = model.fit_predict(coords)

data = locations.copy()

# Map center
center = [
    data['Latitude'].mean(),
    data['Longitude'].mean()
]

# Create map
m = folium.Map(
    location=center,
    zoom_start=13,
    tiles=None
)

# Road map with location names
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, HOT",
    name="Road Map",
    max_zoom=19
).add_to(m)

# Cluster colors
colors = [
    'blue', 'red', 'green', 'purple',
    'orange', 'darkred', 'cadetblue',
    'pink', 'darkgreen'
]

clusters = sorted(data['Cluster'].unique())
cluster_colors = {
    cluster: (
        'black' if cluster == -1
        else colors[i % len(colors)]
    )
    for i, cluster in enumerate(clusters)
}

# Display delivery locations
for _, row in data.iterrows():
    cluster = int(row['Cluster'])
    color = cluster_colors[cluster]

    folium.CircleMarker(
        location=[
            row['Latitude'],
            row['Longitude']
        ],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Cluster: {cluster}"
    ).add_to(m)

# Layer control
folium.LayerControl().add_to(m)

# Save map
m.save("Food_Delivery_DBSCAN_Map.html")

# Show map
display(m)

In [20]:
coords = np.radians(locations[['Latitude', 'Longitude']].to_numpy())
earth_radius_km = 6371.0088
radius_km = 0.35
model = DBSCAN(eps=radius_km / earth_radius_km, min_samples=12,
               algorithm='ball_tree', metric='haversine')
locations['Cluster'] = model.fit_predict(coords)

In [21]:
print('Total clusters:', len(set(locations['Cluster']) - {-1}))
print('Noise points:', (locations['Cluster'] == -1).sum())
print(locations['Cluster'].value_counts().sort_index())

Total clusters: 4
Noise points: 49
Cluster
-1     49
 0    150
 1    150
 2    151
 3    150
Name: count, dtype: int64


In [22]:
delivery_map = folium.Map(location=[11.67, 78.145], zoom_start=13)
colors = ['blue', 'red', 'green', 'purple', 'orange']
for _, row in locations.iterrows():
    cluster = int(row['Cluster'])
    color = 'black' if cluster == -1 else colors[cluster % len(colors)]
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=3, color=color, fill=True,
        fill_color=color, fill_opacity=0.65,
        popup=f'Cluster: {cluster}'
    ).add_to(delivery_map)

In [29]:
delivery_map.save('delivery_hotspots.html')
display(delivery_map)

In [31]:

import folium
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from IPython.display import display
from branca.colormap import linear
from branca.element import Element

# Ensure the Cluster column exists on the locations dataframe
if 'Cluster' not in locations.columns:
    coords = np.radians(locations[['Latitude', 'Longitude']].to_numpy())
    earth_radius_km = 6371.0088
    radius_km = 0.35
    model = DBSCAN(eps=radius_km / earth_radius_km, min_samples=12,
                   algorithm='ball_tree', metric='haversine')
    locations['Cluster'] = model.fit_predict(coords)

data = locations.copy()

# Map center
center = [
    data['Latitude'].mean(),
    data['Longitude'].mean()
]

# Create map
m = folium.Map(
    location=center,
    zoom_start=13,
    tiles=None
)

# Road map with location names
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, HOT",
    name="Road Map",
    max_zoom=19
).add_to(m)

# Cluster colors based on delivery demand
colors = {
    'Highest': 'red',
    'Moderate': 'orange',
    'Below Average': 'purple',
    'Low': 'green',
    'Noise': 'black'
}

# Count orders per cluster
cluster_counts = (
    data[data['Cluster'] != -1]
    .groupby('Cluster')
    .size()
    .sort_values(ascending=False)
)

clusters = sorted(data['Cluster'].unique())


# Classify using actual order counts

mean_orders = cluster_counts.mean()

cluster_levels = {}

for cluster, count in cluster_counts.items():

    ratio = count / mean_orders

    if ratio >= 1.5:
        level = 'Highest'
    elif ratio >= 1.1:
        level = 'Moderate'
    elif ratio >= 0.9:
        level = 'Average'
    elif ratio >= 0.5:
        level = 'Below Average'
    else:
        level = 'Low'

    cluster_levels[cluster] = level

cluster_levels[-1] = 'Noise'

# Add Average color to existing colors dictionary
colors['Average'] = 'blue'

# Display delivery locations
for _, row in data.iterrows():
    cluster = int(row['Cluster'])
    color = cluster_colors[cluster]
    level = cluster_levels[cluster]

    folium.CircleMarker(
        location=[
            row['Latitude'],
            row['Longitude']
        ],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Cluster: {cluster}<br>Demand: {level}"
    ).add_to(m)

# Add legend
legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 200px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 12px;
border-radius: 8px;
">
<b>Food Delivery Demand</b><br><br>
<span style="color:red;">●</span> Highest Orders<br>
<span style="color:orange;">●</span> Moderate Orders<br>
<span style="color:purple;">●</span> Below Average<br>
<span style="color:green;">●</span> Low Orders<br>
<span style="color:black;">●</span> Noise
</div>
"""

m.get_root().html.add_child(Element(legend_html))

# Display cluster summary
print("FOOD DELIVERY CLUSTER ANALYSIS")
print("--------------------------------")

for cluster, count in cluster_counts.items():
    print(
        f"Cluster {cluster} | "
        f"Orders: {count} | "
        f"Demand: {cluster_levels[cluster]}"
    )

# Layer control
folium.LayerControl().add_to(m)

# Save map
m.save("Food_Delivery_DBSCAN_Map.html")

# Show map
display(m)


FOOD DELIVERY CLUSTER ANALYSIS
--------------------------------
Cluster 2 | Orders: 151 | Demand: Average
Cluster 0 | Orders: 150 | Demand: Average
Cluster 1 | Orders: 150 | Demand: Average
Cluster 3 | Orders: 150 | Demand: Average
